# Lab 2C · Ultralytics for beginners

**Computer Vision for Developers · SDAIA Academy** · Day 2 · optional, self-paced

**Goal:** Do the Day 2 ideas in a few lines with YOLO26: predict, read the results, track a video, train on a tiny dataset and prompt YOLOE-26 with text — then see what the library hid from you.

## How to use this notebook

1. Switch the runtime to a **T4 GPU** first: *Runtime → Change runtime type → T4 GPU*.
2. Run the cells **from top to bottom** with *Shift + Enter*. Do not skip cells: later cells use variables from earlier ones.
3. Cells with a **TODO** comment need **one line** from you. The cell right after each TODO is a check: it prints ✅ when your line is right, or a hint when it is not.
4. Nobody writes a cell from empty. If you are stuck for more than five minutes, open the *solved* version of this notebook and compare.
5. Colab forgets files when the session ends. If you come back later, run everything again from the top.
6. The **Extra tasks** at the end are optional and are not graded.

Expected time: about 30 minutes in class; finish the rest at home.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics>=8.4.0", "supervision"], check=False)
import ultralytics
ultralytics.checks()

In [ ]:
# GPU check, with a CPU fallback
import os, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("✅ GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU found. Runtime → Change runtime type → T4 GPU, then Runtime → Restart.")
    print("   You can keep going on CPU: the notebook switches to CPU (YOLO26n is small enough).")
# FAST mode shrinks data and epochs so everything still finishes on a CPU.
FAST = DEVICE == "cpu" or os.environ.get("CVD_FAST") == "1"
print("DEVICE =", DEVICE, "| FAST =", FAST)

> **Licence first.** Ultralytics is **AGPL-3.0**. You may use it freely to learn. If you ship it inside a product, you must open-source your whole application or buy an enterprise licence (slide 2.66). torchvision (BSD), Transformers (Apache 2.0) and supervision (MIT) do not have this condition.

## 5 · One look: predict in one line

`yolo26n.pt` is the nano YOLO26 model, trained on COCO's 80 classes. YOLO26 is **end-to-end**: it outputs one box per object with **no NMS step**, like DETR.

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt
model = YOLO("yolo26n.pt")
results = model("https://ultralytics.com/images/bus.jpg")
r = results[0]
plt.figure(figsize=(6, 8)); plt.imshow(r.plot()[..., ::-1]); plt.axis("off"); plt.show()

### 5.1 · Read the results field by field
`r.boxes.xyxy` (pixels), `r.boxes.conf` (confidence), `r.boxes.cls` (class index), and `r.names` maps index → name.

In [ ]:
print("xyxy:", r.boxes.xyxy.shape, "| conf:", r.boxes.conf.shape, "| cls:", r.boxes.cls.shape)
names = [r.names[int(c)] for c in r.boxes.cls]
for n, c, b in zip(names, r.boxes.conf.tolist(), r.boxes.xyxy.tolist()):
    print(f"{n:<8} {c:.2f}  {[round(v) for v in b]}")
n_people = names.count("person")
print("people:", n_people)

In [ ]:
# Check
try:
    assert len(names) == len(r.boxes) and n_people >= 3
    print('✅ Names read correctly and the people are counted.')
except Exception as e:
    print('❌ Not yet. Hint: names = [r.names[int(c)] for c in r.boxes.cls]')

### 5.2 · Box formats, again
The same boxes in three conventions — Ultralytics gives you all of them.

In [ ]:
print("xyxy  (pixels)    ", [round(v) for v in r.boxes.xyxy[0].tolist()])
print("xywh  (centre, px) ", [round(v) for v in r.boxes.xywh[0].tolist()])
print("xywhn (YOLO, 0–1)  ", [round(v, 3) for v in r.boxes.xywhn[0].tolist()])

## 8 · Tracking in one argument
`model.track(..., tracker="bytetrack.yaml")` runs detection plus ByteTrack and returns IDs in `r.boxes.id`.

In [ ]:
import supervision as sv
from supervision.assets import download_assets, VideoAssets
video = download_assets(VideoAssets.VEHICLES)
ids = set()
N = 40 if FAST else 200
for k, res in enumerate(model.track(video, tracker="bytetrack.yaml", stream=True, verbose=False, imgsz=960)):
    if res.boxes.id is not None:
        ids.update(res.boxes.id.int().tolist())
    if k + 1 >= N:
        break
n_tracks = len(ids)
print(f"{n_tracks} different objects tracked in {N} frames")
plt.figure(figsize=(12, 7)); plt.imshow(res.plot()[..., ::-1]); plt.axis("off"); plt.show()

In [ ]:
# Check
try:
    assert n_tracks == len(ids) and n_tracks > 0
    print('✅ Unique track IDs counted.')
except Exception as e:
    print('❌ Not yet. Hint: n_tracks = len(ids)')

## Train on a tiny dataset (about 2 minutes)
`coco8.yaml` is a built-in dataset of 8 images — far too small to be useful, but perfect for seeing the training loop, curves and metrics end to end.

In [ ]:
train_model = YOLO("yolo26n.pt")
metrics = train_model.train(data="coco8.yaml", epochs=3 if FAST else 20, imgsz=640, batch=8, plots=True, verbose=False)
print("mAP@0.5:", round(metrics.box.map50, 3), "| mAP@0.5:0.95:", round(metrics.box.map, 3))
print("results saved in:", metrics.save_dir)

In [ ]:
# Escape hatch: skip training and evaluate the pretrained model instead
USE_PRETRAINED = False   # ← set to True and run this cell
if USE_PRETRAINED:
    metrics = YOLO("yolo26n.pt").val(data="coco8.yaml", imgsz=640, verbose=False)
    print("pretrained mAP@0.5:", round(metrics.box.map50, 3))

In [ ]:
from PIL import Image
import os
curves = os.path.join(str(metrics.save_dir), "results.png")
if os.path.exists(curves):
    plt.figure(figsize=(14, 7)); plt.imshow(Image.open(curves)); plt.axis("off"); plt.show()

## Open vocabulary: YOLOE-26 with text prompts
Type the classes you want; YOLOE-26 returns boxes **and masks** for them (Day 3, Act 5). The first `set_classes` call downloads a text encoder.

In [ ]:
from ultralytics import YOLOE
yoloe = YOLOE("yoloe-26s-seg.pt")
prompts = ["person", "bus", "backpack"]
yoloe.set_classes(prompts)
er = yoloe.predict("https://ultralytics.com/images/bus.jpg", verbose=False)[0]
print("found:", [er.names[int(c)] for c in er.boxes.cls])
plt.figure(figsize=(6, 8)); plt.imshow(er.plot()[..., ::-1]); plt.axis("off"); plt.show()

In [ ]:
# Check
try:
    assert len(prompts) >= 1 and all(isinstance(p, str) for p in prompts)
    print('✅ YOLOE ran with your own vocabulary.')
except Exception as e:
    print('❌ Not yet. Hint: prompts = ["person", "bus", "backpack"]')

## What Ultralytics hid from you

Every line above hides something you built by hand on Day 2:

| One line here | What it hid | Where you did it by hand |
|---|---|---|
| `model(img)` | resizing, normalising, box decoding, confidence filtering | Lab 2A, torchvision `weights.transforms()` |
| YOLO26 has no NMS | older YOLOs ran NMS with an IoU threshold you could get wrong | Lab 2A, `nms_numpy` |
| `metrics.box.map` | TP/FP matching at IoU ≥ 0.5, PR curve, interpolated AP, mean over classes | Lab 2A, `average_precision` |
| `model.track(...)` | ByteTrack: predict, match with IoU, keep low-confidence boxes | Lab 2B, `ByteTrackTracker` + `LineZone` |
| `train(data=...)` | data format (normalised cxcywh .txt files), augmentation, loss, schedule | Lab 2B, the RT-DETRv2 `Trainer` |

**Use the shortcut when you understand what it does.** And remember the AGPL licence before you put it in a product.

## Extra tasks (optional)

1. **Sizes:** compare `yolo26n.pt` and `yolo26s.pt` on bus.jpg: number of boxes, confidences, and speed (`r.speed`).
2. **Confidence:** run `model(img, conf=0.1)` and `conf=0.7`. What changes?
3. **Export:** export YOLO26n to ONNX with `model.export(format="onnx")` and note the file size.

In [ ]:
# Extra 1
for w in ("yolo26n.pt", "yolo26s.pt"):
    rr = YOLO(w)("https://ultralytics.com/images/bus.jpg", verbose=False)[0]
    print(w, len(rr.boxes), "boxes |", {k: round(v, 1) for k, v in rr.speed.items()})

In [ ]:
# Extra 2
for c in (0.1, 0.7):
    print(c, len(model("https://ultralytics.com/images/bus.jpg", conf=c, verbose=False)[0].boxes), "boxes")

In [ ]:
# Extra 3
path = YOLO("yolo26n.pt").export(format="onnx")
print(path, round(os.path.getsize(path) / 1e6, 1), "MB")